In [1]:
from google.colab import files

uploaded = files.upload()

Saving energy_master.csv to energy_master.csv


In [15]:
price_lag_1 = rl['price'].shift(1).iloc[-1]
load = energy['load'].iloc[-1]
battery_level = battery
state = [
    price,
    price_lag_1,
    load,
    hour,
    battery_level
]

In [16]:
import numpy as np

dqn_df = energy[['price','load']].copy()

dqn_df['price_lag_1'] = dqn_df['price'].shift(1)
dqn_df['hour'] = dqn_df.index.hour

dqn_df = dqn_df.dropna()

In [17]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

features = ['price','price_lag_1','load','hour']

dqn_df[features] = scaler.fit_transform(dqn_df[features])

In [18]:
#Build DQN Network
import torch
import torch.nn as nn

class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )

    def forward(self, x):
        return self.net(x)

In [24]:
#Initialize model
input_dim = 4
output_dim = 3

policy_net = DQN(input_dim, output_dim)
target_net = DQN(input_dim, output_dim)

target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [20]:
#Experience Replay Buffer
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.array, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [26]:
#Agent + target network + optimizer.
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

policy_net = DQN(input_dim, output_dim).to(device)
target_net = DQN(input_dim, output_dim).to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

replay = ReplayBuffer()

In [27]:
#Hyperparameters
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

batch_size = 64
target_update = 10

In [28]:
#FULL TRAINING LOOP
profits = []

for episode in range(50):

    state = dqn_df.iloc[0][features].values.astype(np.float32)
    battery = 50
    total_reward = 0

    for t in range(len(dqn_df)-1):

        # state tensor
        state_t = torch.FloatTensor(state).to(device)

        # epsilon-greedy
        if np.random.rand() < epsilon:
            action = np.random.randint(3)
        else:
            with torch.no_grad():
                action = policy_net(state_t).argmax().item()

        price = dqn_df.iloc[t]['price']

        # battery dynamics
        reward = 0
        if action == 1:
            reward = -price * 10
        elif action == 2:
            reward = price * 10

        next_state = dqn_df.iloc[t+1][features].values.astype(np.float32)

        done = (t == len(dqn_df)-2)

        replay.push(state, action, reward, next_state, done)

        state = next_state
        total_reward += reward

        # TRAIN STEP
        if len(replay) > batch_size:

            s, a, r, s2, d = replay.sample(batch_size)

            s = torch.FloatTensor(s).to(device)
            a = torch.LongTensor(a).to(device)
            r = torch.FloatTensor(r).to(device)
            s2 = torch.FloatTensor(s2).to(device)
            d = torch.FloatTensor(d).to(device)

            q_values = policy_net(s).gather(1, a.unsqueeze(1)).squeeze()

            next_q = target_net(s2).max(1)[0]
            target = r + gamma * next_q * (1 - d)

            loss = loss_fn(q_values, target.detach())

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # update epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

    # update target network
    if episode % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())

    profits.append(total_reward)

    print(f"Episode {episode} | Profit {total_reward:.2f} | ε={epsilon:.3f}")

Episode 0 | Profit -492.77 | ε=0.995
Episode 1 | Profit 774.90 | ε=0.990
Episode 2 | Profit 836.29 | ε=0.985
Episode 3 | Profit 1064.85 | ε=0.980
Episode 4 | Profit 2226.00 | ε=0.975
Episode 5 | Profit 1589.13 | ε=0.970
Episode 6 | Profit 2055.24 | ε=0.966
Episode 7 | Profit 3045.89 | ε=0.961
Episode 8 | Profit 2966.85 | ε=0.956
Episode 9 | Profit 3101.34 | ε=0.951
Episode 10 | Profit 3654.48 | ε=0.946
Episode 11 | Profit 4555.41 | ε=0.942
Episode 12 | Profit 4470.32 | ε=0.937
Episode 13 | Profit 4426.08 | ε=0.932
Episode 14 | Profit 4897.98 | ε=0.928
Episode 15 | Profit 4723.76 | ε=0.923
Episode 16 | Profit 5046.10 | ε=0.918
Episode 17 | Profit 5469.72 | ε=0.914
Episode 18 | Profit 6295.60 | ε=0.909
Episode 19 | Profit 6871.68 | ε=0.905
Episode 20 | Profit 7181.60 | ε=0.900
Episode 21 | Profit 7360.44 | ε=0.896
Episode 22 | Profit 8225.23 | ε=0.891
Episode 23 | Profit 7804.79 | ε=0.887
Episode 24 | Profit 8028.85 | ε=0.882
Episode 25 | Profit 9156.81 | ε=0.878
Episode 26 | Profit 9218

Market timing learning
buy low price regimes
sell high price regimes